# TrustCV UCI HAR: sitting vs standing grouped-leakage extension

This notebook extends the earlier UCI HAR real-data illustration. The original **dynamic vs static** task produced near-ceiling performance for every validation design, so the leakage checker was correct but the performance contrast was visually weak.

Here the default task is **sitting vs standing**, a harder static-posture contrast. The purpose is to test whether record-wise K-Fold looks better than subject-aware validation when the same subjects appear in both training and validation folds.

The validation workflow uses TrustCV objects for:

- CV splitters: `KFold`, `GroupKFold`, `StratifiedGroupKFold`, `LeaveOneGroupOut`, `NestedGroupedCV`
- CV execution: `UniversalCVRunner`
- fold logging: `LeakageDetectionCallback`, `ClassDistributionLogger`
- leakage checks: `DataLeakageChecker`
- clinical metrics: `ClinicalMetrics`, `oob_clinical_metrics`
- grouped-CV visualization: `plot_grouped_cv`

Standard Python, NumPy, pandas, and matplotlib are used only for data loading, formatting, and visualization. The predictive estimator is a sklearn-compatible `RandomForestClassifier`, which TrustCV evaluates through its sklearn adapter.

## Cell 1 — Runtime settings

Start with `SMOKE_MODE=True` to test the notebook. For final manuscript numbers, set `SMOKE_MODE=False` and rerun from the top.

In [ ]:
INSTALL_SOURCE = "pypi"  # "pypi", "github", or "none"

# Main recommended sensitivity task.
TASK_MODE = "sitting_vs_standing"  # "sitting_vs_standing", "dynamic_vs_static", "upstairs_vs_downstairs"

SMOKE_MODE = True
RANDOM_STATE = 42

DATA_DIR = "/content/uci_har_data"
OUTPUT_DIR = "/content/trustcv_har_sitting_standing_outputs"

# If automatic download fails, upload either the extracted folder or the zip to Colab.
# Example extracted folder: LOCAL_HAR_ROOT = "/content/UCI HAR Dataset"
# Example zip file:        LOCAL_ZIP_PATH = "/content/UCI HAR Dataset.zip"
LOCAL_HAR_ROOT = None
LOCAL_ZIP_PATH = None

# CV/model size. Use final settings only after the smoke run succeeds.
N_SPLITS = 4 if SMOKE_MODE else 10
N_ESTIMATORS = 80 if SMOKE_MODE else 200
NESTED_INNER_SPLITS = 3 if SMOKE_MODE else 5
RUN_LOGO = True
RUN_NESTED_DIAGNOSTIC = True  # keep True; smoke mode uses a smaller nested configuration

# Near-duplicate threshold. For engineered HAR features, 0.92 is more sensitive than 0.99.
NEAR_DUPLICATE_THRESHOLD = 0.92

print({
    "TASK_MODE": TASK_MODE,
    "SMOKE_MODE": SMOKE_MODE,
    "N_SPLITS": N_SPLITS,
    "N_ESTIMATORS": N_ESTIMATORS,
    "RUN_LOGO": RUN_LOGO,
    "NESTED_INNER_SPLITS": NESTED_INNER_SPLITS,
    "RUN_NESTED_DIAGNOSTIC": RUN_NESTED_DIAGNOSTIC,
})

## Cell 2 — Install TrustCV

Use PyPI for reproducibility. Use GitHub only if you intentionally want the newest development version.

In [ ]:
import subprocess
import sys


def pip_install(*packages):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *packages]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

if INSTALL_SOURCE == "pypi":
    pip_install("trustcv>=1.0.7", "scikit-learn>=1.3,<1.8", "pandas", "matplotlib", "tabulate")
elif INSTALL_SOURCE == "github":
    pip_install("git+https://github.com/ki-smile/trustcv.git", "scikit-learn>=1.3,<1.8", "pandas", "matplotlib", "tabulate")
elif INSTALL_SOURCE == "none":
    print("Skipping installation. Make sure trustcv is already importable.")
else:
    raise ValueError("INSTALL_SOURCE must be 'pypi', 'github', or 'none'.")

import trustcv
print("trustcv version:", getattr(trustcv, "__version__", "unknown"))

## Cell 3 — Imports

TrustCV handles splitting, running, leakage checking, and clinical metrics. sklearn is used only for the model estimator that TrustCV evaluates.

In [ ]:
from pathlib import Path
import io
import os
import urllib.request
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from trustcv import (
    KFold,
    GroupKFold,
    StratifiedGroupKFold,
    LeaveOneGroupOut,
    NestedGroupedCV,
    UniversalCVRunner,
    LeakageDetectionCallback,
    ClassDistributionLogger,
    DataLeakageChecker,
    ClinicalMetrics,
)
from trustcv.metrics import oob_clinical_metrics
from trustcv.validators import NestedGroupedCV as TrustCVNestedGroupedCV
from trustcv.visualization import plot_grouped_cv

from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_STATE)

## Cell 4 — Define HAR task options

The default target is sitting vs standing. Dynamic vs static remains available as a comparison, but it usually saturates near 100% performance.

In [ ]:
# UCI HAR activity codes:
# 1 WALKING, 2 WALKING_UPSTAIRS, 3 WALKING_DOWNSTAIRS, 4 SITTING, 5 STANDING, 6 LAYING
TASKS = {
    "sitting_vs_standing": {
        "positive_codes": (4,),
        "negative_codes": (5,),
        "positive_name": "sitting",
        "negative_name": "standing",
        "label": "sitting (1) vs standing (0)",
    },
    "dynamic_vs_static": {
        "positive_codes": (1, 2, 3),
        "negative_codes": (4, 5, 6),
        "positive_name": "dynamic",
        "negative_name": "static",
        "label": "dynamic (1) vs static (0)",
    },
    "upstairs_vs_downstairs": {
        "positive_codes": (2,),
        "negative_codes": (3,),
        "positive_name": "upstairs",
        "negative_name": "downstairs",
        "label": "upstairs (1) vs downstairs (0)",
    },
}

if TASK_MODE not in TASKS:
    raise ValueError(f"Unknown TASK_MODE={TASK_MODE}. Choose from {list(TASKS)}")

TASKS[TASK_MODE]

## Cell 5 — Download or locate UCI HAR

The code first checks for a local folder or uploaded zip. If neither exists, it downloads the public UCI HAR archive.

In [ ]:
UCI_URLS = [
    "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip",
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00240/UCI%20HAR%20Dataset.zip",
]


def extract_har_zip(zip_bytes, target_dir):
    target_dir = Path(target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        names = zf.namelist()
        inner_zip_names = [n for n in names if n.lower().endswith(".zip")]
        if inner_zip_names:
            inner_bytes = zf.read(inner_zip_names[0])
            with zipfile.ZipFile(io.BytesIO(inner_bytes)) as inner:
                inner.extractall(target_dir)
        else:
            zf.extractall(target_dir)


def prepare_har_root():
    if LOCAL_HAR_ROOT is not None:
        root = Path(LOCAL_HAR_ROOT)
        if root.exists():
            print("Using local extracted HAR folder:", root)
            return root
        raise FileNotFoundError(f"LOCAL_HAR_ROOT does not exist: {root}")

    data_dir = Path(DATA_DIR)
    data_dir.mkdir(parents=True, exist_ok=True)
    root = data_dir / "UCI HAR Dataset"
    if root.exists():
        print("Using existing extracted dataset:", root)
        return root

    if LOCAL_ZIP_PATH is not None:
        zip_path = Path(LOCAL_ZIP_PATH)
        if not zip_path.exists():
            raise FileNotFoundError(f"LOCAL_ZIP_PATH does not exist: {zip_path}")
        print("Extracting uploaded zip:", zip_path)
        extract_har_zip(zip_path.read_bytes(), data_dir)
        return root

    last_error = None
    for url in UCI_URLS:
        try:
            print("Downloading:", url)
            with urllib.request.urlopen(url, timeout=60) as response:
                zip_bytes = response.read()
            extract_har_zip(zip_bytes, data_dir)
            if root.exists():
                print("Extracted dataset:", root)
                return root
        except Exception as exc:
            print("Download attempt failed:", exc)
            last_error = exc

    raise RuntimeError(
        "Could not download/extract UCI HAR. Upload 'UCI HAR Dataset.zip' to Colab and set LOCAL_ZIP_PATH."
    ) from last_error

har_root = prepare_har_root()
print("HAR root:", har_root)

## Cell 6 — Load UCI HAR and recast the binary target

We merge the original train/test files and then let TrustCV create new validation splits. The subject ID becomes the `groups` variable.

In [ ]:
def load_har_binary(har_root, task_mode):
    har_root = Path(har_root)
    cfg = TASKS[task_mode]
    pos_codes = cfg["positive_codes"]
    neg_codes = cfg["negative_codes"]

    X_parts, y_parts, subject_parts, activity_parts, split_parts = [], [], [], [], []
    for split in ["train", "test"]:
        X_split = np.loadtxt(har_root / split / f"X_{split}.txt")
        y_multi = np.loadtxt(har_root / split / f"y_{split}.txt").astype(int)
        subjects = np.loadtxt(har_root / split / f"subject_{split}.txt").astype(int)

        keep = np.isin(y_multi, pos_codes + neg_codes)
        X_parts.append(X_split[keep])
        y_parts.append(np.isin(y_multi[keep], pos_codes).astype(int))
        subject_parts.append(subjects[keep])
        activity_parts.append(y_multi[keep])
        split_parts.append(np.repeat(split, int(keep.sum())))

    X = np.vstack(X_parts)
    y = np.concatenate(y_parts)
    groups = np.concatenate(subject_parts)
    activity_code = np.concatenate(activity_parts)
    source_split = np.concatenate(split_parts)

    meta = pd.DataFrame({
        "subject": groups,
        "activity_code": activity_code,
        "source_split": source_split,
        "y": y,
        "class_label": np.where(y == 1, cfg["positive_name"], cfg["negative_name"]),
    })
    return X, y, groups, meta

X, y, groups, meta = load_har_binary(har_root, TASK_MODE)

print("Task:", TASKS[TASK_MODE]["label"])
print("X shape:", X.shape)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Subjects:", np.unique(groups).size)
display(meta.head())

## Cell 7 — Basic dataset checks and visualization

These checks confirm the repeated-measures structure. The key issue is that many windows belong to the same subject.

In [ ]:
summary = pd.DataFrame({
    "quantity": [
        "windows",
        "features",
        "subjects",
        "positive prevalence",
        "missing values",
    ],
    "value": [
        X.shape[0],
        X.shape[1],
        np.unique(groups).size,
        float(np.mean(y)),
        int(np.isnan(X).sum()),
    ],
})
display(summary)

class_table = meta["class_label"].value_counts().rename_axis("class").reset_index(name="n_windows")
display(class_table)

subject_table = pd.crosstab(meta["subject"], meta["class_label"])
display(subject_table.describe())

fig, ax = plt.subplots(figsize=(5, 4))
class_table.set_index("class")["n_windows"].plot(kind="bar", ax=ax)
ax.set_title(f"Class balance: {TASK_MODE}")
ax.set_ylabel("Number of windows")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / f"{TASK_MODE}_class_balance.png", dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
subject_table.plot(kind="bar", stacked=True, ax=ax, width=0.9)
ax.set_title("Per-subject window counts")
ax.set_xlabel("Subject ID")
ax.set_ylabel("Number of windows")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / f"{TASK_MODE}_subject_counts.png", dpi=300)
plt.show()

## Cell 8 — Define TrustCV splitters and model factory

The first splitter is the leakage-blind comparator. The remaining splitters keep subjects disjoint between train and validation.

In [ ]:
def make_model():
    return RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced",
    )

cv_methods = {
    "KFold record-wise": KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
    "GroupKFold": GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
    "StratifiedGroupKFold": StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE),
}

if RUN_LOGO:
    cv_methods["LeaveOneGroupOut"] = LeaveOneGroupOut()

for name, cv in cv_methods.items():
    try:
        n = cv.get_n_splits(X, y, groups)
    except Exception:
        n = cv.get_n_splits()
    print(f"{name}: {n} folds")

## Cell 9 — Visualize TrustCV grouped split behavior

`plot_grouped_cv` comes from TrustCV. For K-Fold, subjects appear in multiple validation folds. For GroupKFold, subjects are held out as groups.

In [ ]:
# KFold and grouped allocation plots. These are diagnostic, not model results.
for plot_name in ["KFold record-wise", "GroupKFold"]:
    try:
        ax = plot_grouped_cv(
            cv_methods[plot_name],
            groups=groups,
            n_splits=N_SPLITS,
            figsize=(10, 4),
            title=f"TrustCV split visualization: {plot_name}",
        )
        plt.tight_layout()
        plt.savefig(Path(OUTPUT_DIR) / f"{TASK_MODE}_{plot_name.replace(' ', '_')}_trustcv_split.png", dpi=300)
        plt.show()
    except Exception as exc:
        print(f"Could not plot {plot_name}: {exc}")

## Cell 10 — Helper functions using TrustCV results

The main pooled clinical metrics come from `oob_clinical_metrics`. The subject-macro table is built by applying TrustCV `ClinicalMetrics` within each subject and then averaging the subject-wise metrics.

In [ ]:
def extract_oob_arrays(cv_result, y):
    """Reconstruct out-of-fold labels, predictions, and positive-class probabilities from TrustCV CVResults."""
    y = np.asarray(y)
    y_true_all, y_pred_all, y_proba_all, subject_index_all = [], [], [], []

    for fold_id, ((train_idx, test_idx), fold_scores) in enumerate(zip(cv_result.indices, cv_result.scores), start=1):
        y_true_all.append(y[test_idx])
        subject_index_all.append(np.asarray(test_idx))

        pred = fold_scores.get("predictions")
        if pred is None:
            pred = fold_scores.get("y_pred")
        y_pred_all.append(np.asarray(pred).ravel())

        prob = fold_scores.get("probabilities")
        if prob is None:
            prob = fold_scores.get("y_proba")
        prob = np.asarray(prob)
        if prob.ndim == 2 and prob.shape[1] > 1:
            prob = prob[:, 1]
        y_proba_all.append(prob.ravel())

    test_indices = np.concatenate(subject_index_all)
    return {
        "test_indices": test_indices,
        "y_true": np.concatenate(y_true_all),
        "y_pred": np.concatenate(y_pred_all),
        "y_proba": np.concatenate(y_proba_all),
    }


def subject_macro_clinical_metrics(cv_result, y, groups):
    """Apply TrustCV ClinicalMetrics within each subject, then average subject-level values."""
    oob = extract_oob_arrays(cv_result, y)
    subject_ids = np.asarray(groups)[oob["test_indices"]]
    clinical = ClinicalMetrics(confidence_level=0.95, prevalence=float(np.mean(y)))

    rows = []
    for subject_id in np.sort(np.unique(subject_ids)):
        m = subject_ids == subject_id
        if len(np.unique(oob["y_true"][m])) < 2:
            continue
        metrics = clinical.calculate_all(
            y_true=oob["y_true"][m],
            y_pred=oob["y_pred"][m],
            y_proba=oob["y_proba"][m],
        )
        rows.append({
            "subject": int(subject_id),
            "auc_roc": float(metrics.get("auc_roc", np.nan)),
            "sensitivity": float(metrics.get("sensitivity", np.nan)),
            "specificity": float(metrics.get("specificity", np.nan)),
            "youdens_index": float(metrics.get("youdens_index", np.nan)),
            "accuracy": float(metrics.get("accuracy", np.nan)),
        })

    subject_df = pd.DataFrame(rows)
    macro = subject_df.drop(columns=["subject"]).mean(numeric_only=True).to_dict()
    macro_sd = subject_df.drop(columns=["subject"]).std(numeric_only=True).add_suffix("_sd").to_dict()
    return {**macro, **macro_sd, "n_subjects": int(subject_df.shape[0])}, subject_df


def run_trustcv_method(method_name, splitter):
    """Run one TrustCV CV method with TrustCV callbacks and TrustCV OOB clinical metrics."""
    print("=" * 80)
    print(method_name)
    print("=" * 80)

    runner = UniversalCVRunner(cv_splitter=splitter, framework="sklearn", verbose=1)
    callbacks = [
        LeakageDetectionCallback(data=(X, y), groups=groups, verbose=1),
        ClassDistributionLogger(
            labels=y,
            label_names={0: TASKS[TASK_MODE]["negative_name"], 1: TASKS[TASK_MODE]["positive_name"]},
            verbose=0,
        ),
    ]

    cv_result = runner.run(
        model=make_model,
        data=(X, y, groups),
        groups=groups,
        metrics=["accuracy", "balanced_accuracy", "f1", "precision", "recall", "roc_auc"],
        callbacks=callbacks,
    )

    clinical = ClinicalMetrics(confidence_level=0.95, prevalence=float(np.mean(y)))
    pooled = oob_clinical_metrics(cv_result, y, clinical=clinical)
    macro, subject_df = subject_macro_clinical_metrics(cv_result, y, groups)

    print(cv_result.summary())
    print("Pooled OOB clinical metrics from trustcv.metrics.oob_clinical_metrics:")
    for key in ["auc_roc", "sensitivity", "specificity", "youdens_index", "accuracy"]:
        print(f"  {key}: {pooled.get(key, np.nan):.4f}")
    print("Subject-macro clinical metrics using trustcv.ClinicalMetrics per subject:")
    for key in ["auc_roc", "sensitivity", "specificity", "youdens_index", "accuracy"]:
        print(f"  {key}: {macro.get(key, np.nan):.4f}")

    return {
        "method": method_name,
        "splitter": splitter,
        "cv_result": cv_result,
        "pooled": pooled,
        "subject_macro": macro,
        "subject_df": subject_df,
    }

## Cell 11 — Run TrustCV validation methods

This is the main experiment. Watch the TrustCV leakage messages: record-wise K-Fold should be flagged, while grouped methods should pass.

In [ ]:
trustcv_results = {}
for method_name, splitter in cv_methods.items():
    trustcv_results[method_name] = run_trustcv_method(method_name, splitter)

## Cell 12 — Explicit leakage audit with `DataLeakageChecker`

This cell checks subject overlap using TrustCV `DataLeakageChecker.check_cv_splits` on the first fold of each method.

In [ ]:
def explicit_subject_leakage_audit(method_name, splitter):
    checker = DataLeakageChecker(verbose=False)
    train_idx, test_idx = next(iter(splitter.split(X, y, groups=groups)))
    report = checker.check_cv_splits(
        X[train_idx],
        X[test_idx],
        y_train=y[train_idx],
        y_test=y[test_idx],
        patient_ids_train=groups[train_idx],
        patient_ids_test=groups[test_idx],
    )
    shared_subjects = sorted(set(groups[train_idx]) & set(groups[test_idx]))
    return {
        "CV method": method_name,
        "train_subjects": len(set(groups[train_idx])),
        "validation_subjects": len(set(groups[test_idx])),
        "shared_subjects": len(shared_subjects),
        "leakage_types": ", ".join(report.leakage_types) if report.leakage_types else "none",
        "severity": report.severity,
        "verdict": "FLAG" if "patient" in report.leakage_types else "PASS",
    }

leakage_table = pd.DataFrame([
    explicit_subject_leakage_audit(name, splitter)
    for name, splitter in cv_methods.items()
])

display(leakage_table)
leakage_table.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_subject_leakage_audit.csv", index=False)

## Cell 13 — Near-duplicate audit from overlapping HAR windows

The HAR windows overlap by design. This cell uses TrustCV `DataLeakageChecker.check_near_duplicates` to compare record-wise K-Fold versus GroupKFold on the first fold.

In [ ]:
def explicit_near_duplicate_audit():
    checker = DataLeakageChecker(verbose=False)
    rows = []
    for method_name in ["KFold record-wise", "GroupKFold"]:
        splitter = cv_methods[method_name]
        train_idx, test_idx = next(iter(splitter.split(X, y, groups=groups)))
        report = checker.check_near_duplicates(
            X[train_idx],
            X[test_idx],
            similarity_threshold=NEAR_DUPLICATE_THRESHOLD,
        )
        rows.append({
            "CV method": method_name,
            "similarity_threshold": NEAR_DUPLICATE_THRESHOLD,
            "has_leakage": report.get("has_leakage"),
            "near_duplicate_count": report.get("near_duplicate_count"),
            "near_duplicate_percentage": report.get("near_duplicate_percentage"),
        })
    return pd.DataFrame(rows)

near_duplicate_table = explicit_near_duplicate_audit()
display(near_duplicate_table)
near_duplicate_table.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_near_duplicate_audit.csv", index=False)

## Cell 14 — Build manuscript-style table

The table reports pooled observation-level metrics and subject-macro metrics. The leakage verdict comes from TrustCV's explicit leakage checker.

In [ ]:
def fmt_metric(value, digits=3):
    if value is None or pd.isna(value):
        return "NA"
    return f"{float(value):.{digits}f}"


def fmt_ci(value, ci, digits=3):
    if value is None or pd.isna(value):
        return "NA"
    if ci is None or len(ci) != 2:
        return fmt_metric(value, digits)
    return f"{float(value):.{digits}f} ({float(ci[0]):.{digits}f}, {float(ci[1]):.{digits}f})"

reference_method = "GroupKFold"
reference_obs_auc = trustcv_results[reference_method]["pooled"].get("auc_roc")
reference_subj_auc = trustcv_results[reference_method]["subject_macro"].get("auc_roc")

rows = []
for method_name, result in trustcv_results.items():
    pooled = result["pooled"]
    macro = result["subject_macro"]
    leak = leakage_table.loc[leakage_table["CV method"] == method_name].iloc[0]
    rows.append({
        "CV method": method_name,
        "Obs. AUC (95% CI)": fmt_ci(pooled.get("auc_roc"), pooled.get("auc_roc_ci")),
        "Obs. Sens.": fmt_metric(pooled.get("sensitivity")),
        "Obs. Spec.": fmt_metric(pooled.get("specificity")),
        "Obs. Youden J": fmt_metric(pooled.get("youdens_index")),
        "Obs. Accuracy": fmt_metric(pooled.get("accuracy")),
        "Subj.-macro AUC": fmt_metric(macro.get("auc_roc")),
        "Subj.-macro Sens.": fmt_metric(macro.get("sensitivity")),
        "Subj.-macro Spec.": fmt_metric(macro.get("specificity")),
        "Subj.-macro Youden J": fmt_metric(macro.get("youdens_index")),
        "Δ Obs. AUC vs GroupKFold": "Ref" if method_name == reference_method else fmt_metric(pooled.get("auc_roc") - reference_obs_auc),
        "Δ Subj. AUC vs GroupKFold": "Ref" if method_name == reference_method else fmt_metric(macro.get("auc_roc") - reference_subj_auc),
        "Leakage checker": leak["verdict"],
        "Shared subjects in first fold": int(leak["shared_subjects"]),
    })

manuscript_table = pd.DataFrame(rows)
display(manuscript_table)

manuscript_table.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_manuscript_table.csv", index=False)
with open(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_manuscript_table.md", "w", encoding="utf-8") as f:
    f.write(manuscript_table.to_markdown(index=False))

## Cell 15 — Performance and leakage plots

These visualizations are for checking the message of the result before moving it into the manuscript.

In [ ]:
plot_df = []
for method_name, result in trustcv_results.items():
    plot_df.append({
        "CV method": method_name,
        "Observation AUC": result["pooled"].get("auc_roc"),
        "Subject-macro AUC": result["subject_macro"].get("auc_roc"),
    })
plot_df = pd.DataFrame(plot_df).set_index("CV method")

fig, ax = plt.subplots(figsize=(9, 5))
plot_df.plot(kind="bar", ax=ax)
ax.set_ylim(max(0.5, float(plot_df.min().min()) - 0.05), 1.01)
ax.set_ylabel("AUC")
ax.set_title(f"TrustCV HAR validation comparison: {TASK_MODE}")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_auc_comparison.png", dpi=300)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
leakage_table.set_index("CV method")["shared_subjects"].plot(kind="bar", ax=ax)
ax.set_ylabel("Shared subjects in first fold")
ax.set_title("TrustCV subject-overlap leakage audit")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_subject_overlap.png", dpi=300)
plt.show()

## Cell 16 — TrustCV `NestedGroupedCV` with inner grouped tuning

This cell adds the missing nested grouped cross-validation component. It uses the TrustCV nested grouped validator, with an outer group-disjoint evaluation loop and an inner group-disjoint tuning loop. The native TrustCV nested object returns fold-level ROC AUC, balanced accuracy, AUPRC, F1, and best hyperparameters. It does not expose full out-of-fold predictions, so sensitivity, specificity, and subject-macro clinical metrics are not computed for the nested row in the same way as the `UniversalCVRunner` methods.

In [ ]:
if RUN_NESTED_DIAGNOSTIC:
    print("Running TrustCV NestedGroupedCV with inner grouped tuning...")
    print({
        "outer_splits": N_SPLITS,
        "inner_splits": NESTED_INNER_SPLITS,
        "scoring": "roc_auc",
        "mode": "group",
    })

    nested_param_grid = {
        "max_features": ["sqrt", "log2"],
        "min_samples_leaf": [1] if SMOKE_MODE else [1, 2],
    }

    # This is the TrustCV validator-level NestedGroupedCV. It gives ROC AUC and
    # fold-level nested diagnostics. The top-level splitter-style NestedGroupedCV
    # is also available, but it reports only accuracy and best parameters.
    trustcv_nested = TrustCVNestedGroupedCV(
        n_splits_outer=N_SPLITS,
        n_splits_inner=NESTED_INNER_SPLITS,
        scoring="roc_auc",
        mode="group",
    )

    nested_per_fold, nested_summary = trustcv_nested.fit_predict(
        estimator=make_model(),
        X=X,
        y=y,
        groups=groups,
        param_grid=nested_param_grid,
    )

    print("Nested grouped per-fold results:")
    display(nested_per_fold)
    print("Nested grouped summary:")
    display(nested_summary)

    # Leakage audit on the outer grouped split. This should PASS because the
    # outer folds are group-disjoint.
    nested_leakage = explicit_subject_leakage_audit("NestedGroupedCV", trustcv_nested.outer_cv)
    print("Nested outer-fold leakage audit:")
    display(pd.DataFrame([nested_leakage]))

    def bootstrap_mean_ci(values, n_bootstrap=1000, ci=0.95, random_state=RANDOM_STATE):
        """Small reporting helper: bootstrap CI for the mean over outer folds."""
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            return (np.nan, np.nan)
        rng = np.random.default_rng(random_state)
        boot = []
        for _ in range(n_bootstrap):
            boot.append(np.mean(rng.choice(values, size=values.size, replace=True)))
        alpha = 1 - ci
        return (
            float(np.percentile(boot, 100 * alpha / 2)),
            float(np.percentile(boot, 100 * (1 - alpha / 2))),
        )

    nested_auc_mean = float(nested_per_fold["roc_auc"].mean())
    nested_auc_sd = float(nested_per_fold["roc_auc"].std())
    nested_auc_ci = bootstrap_mean_ci(nested_per_fold["roc_auc"].to_numpy())
    nested_bal_acc_mean = float(nested_per_fold["balanced_accuracy"].mean())
    nested_f1_mean = float(nested_per_fold["f1"].mean())
    nested_auprc_mean = float(nested_per_fold["auprc"].mean())

    nested_summary_table = pd.DataFrame([{
        "CV method": "NestedGroupedCV",
        "Outer folds": int(nested_per_fold.shape[0]),
        "Inner folds": int(NESTED_INNER_SPLITS),
        "Outer ROC AUC mean": nested_auc_mean,
        "Outer ROC AUC SD": nested_auc_sd,
        "Outer ROC AUC 95% CI low": nested_auc_ci[0],
        "Outer ROC AUC 95% CI high": nested_auc_ci[1],
        "Outer balanced accuracy mean": nested_bal_acc_mean,
        "Outer AUPRC mean": nested_auprc_mean,
        "Outer F1 mean": nested_f1_mean,
        "Inner scoring": "roc_auc",
        "Tuned parameters": str(nested_param_grid),
        "Leakage checker": nested_leakage["verdict"],
        "Shared subjects in first outer fold": int(nested_leakage["shared_subjects"]),
    }])

    print("Nested grouped diagnostic table:")
    display(nested_summary_table)

    # Optional compact row for the manuscript-style table. Sensitivity and
    # specificity are marked NA because native NestedGroupedCV returns fold-level
    # summary metrics but not full out-of-fold predictions.
    nested_row = {
        "CV method": "NestedGroupedCV",
        "Obs. AUC (95% CI)": fmt_ci(nested_auc_mean, nested_auc_ci),
        "Obs. Sens.": "NA",
        "Obs. Spec.": "NA",
        "Obs. Youden J": "NA",
        "Obs. Accuracy": fmt_metric(nested_bal_acc_mean),
        "Subj.-macro AUC": "NA",
        "Subj.-macro Sens.": "NA",
        "Subj.-macro Spec.": "NA",
        "Subj.-macro Youden J": "NA",
        "Δ Obs. AUC vs GroupKFold": fmt_metric(nested_auc_mean - reference_obs_auc),
        "Δ Subj. AUC vs GroupKFold": "NA",
        "Leakage checker": nested_leakage["verdict"],
        "Shared subjects in first fold": int(nested_leakage["shared_subjects"]),
    }

    manuscript_table_with_nested = pd.concat(
        [manuscript_table, pd.DataFrame([nested_row])],
        ignore_index=True,
    )
    print("Manuscript-style table with NestedGroupedCV row:")
    display(manuscript_table_with_nested)

    # Save nested outputs.
    nested_per_fold.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_nested_grouped_per_fold.csv", index=False)
    nested_summary.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_nested_grouped_summary.csv")
    nested_summary_table.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_nested_grouped_diagnostic.csv", index=False)
    manuscript_table_with_nested.to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_manuscript_table_with_nested.csv", index=False)
    with open(Path(OUTPUT_DIR) / f"{TASK_MODE}_trustcv_manuscript_table_with_nested.md", "w", encoding="utf-8") as f:
        f.write(manuscript_table_with_nested.to_markdown(index=False))
else:
    print("NestedGroupedCV skipped. Set RUN_NESTED_DIAGNOSTIC=True in Cell 1 to run it.")

## Cell 17 — Export all outputs

This creates a zip file containing tables and figures for download from Colab.

In [ ]:
import shutil

# Save per-subject metrics for each method.
for method_name, result in trustcv_results.items():
    safe_name = method_name.lower().replace(" ", "_").replace("-", "_")
    result["subject_df"].to_csv(Path(OUTPUT_DIR) / f"{TASK_MODE}_{safe_name}_subject_metrics.csv", index=False)

zip_base = f"/content/trustcv_har_{TASK_MODE}_outputs"
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Output directory:", OUTPUT_DIR)
print("Zip file:", zip_path)
print("Files:")
for p in sorted(Path(OUTPUT_DIR).glob("*")):
    print("-", p.name)

## Cell 18 — Interpretation scaffold after the run

Use this after you inspect the actual numbers. The key point is whether KFold is flagged for subject leakage and whether the harder task reduces the ceiling effect.

In [ ]:
kfold = manuscript_table.loc[manuscript_table["CV method"] == "KFold record-wise"].iloc[0]
gkf = manuscript_table.loc[manuscript_table["CV method"] == "GroupKFold"].iloc[0]

print("Interpretation scaffold:")
print(
    f"On the UCI HAR {TASK_MODE} task, TrustCV flagged record-wise K-Fold for subject overlap "
    f"({kfold['Shared subjects in first fold']} shared subjects in the first audited fold), "
    f"whereas GroupKFold preserved subject exclusivity "
    f"({gkf['Shared subjects in first fold']} shared subjects). "
    f"The pooled observation-level AUC was {kfold['Obs. AUC (95% CI)']} for record-wise K-Fold "
    f"and {gkf['Obs. AUC (95% CI)']} for GroupKFold. "
    f"This harder static-posture contrast is intended to reduce the ceiling effect seen in the dynamic-vs-static task."
)